In [5]:
%%capture

import altair as alt
import gcsfs
import pandas as pd

#from calitp_data_analysis import calitp_color_palette as cp
from IPython.display import HTML, Markdown, display
#from update_vars import GCS_FILE_PATH, MONTH, PUBLIC_FILENAME, YEAR
#from _01_ntd_ridership_utils import sum_by_group
from gtfs_curator_utils import magics

GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

alt.data_transformers.enable("vegafusion")

WIDTH = 300
HEIGHT = 150

In [6]:
# parameters cell for local
rtpa = "Metropolitan Transportation Commission"

In [7]:
%%capture_parameters
rtpa

{"rtpa": "Metropolitan Transportation Commission"}


# {rtpa}
Annual Ridership Trends

Download data from our **[public folder](https://console.cloud.google.com/storage/browser/calitp-publish-data-analysis)** by navigating to `ntd_annual_ridership` and selecting a file.

Transit operators/agencies that submit annual reports to NTD are included in this report. Reporters that were previously active reporters, but are currently not, may appear. This may result in Reporters showing zero or partial ridership data in the report.

If a Reporter, type of service, mode, or any combination of, is not a annual reporter or has not reported data since 2018, they will not appear in the report.

Examples:

* **Reporter A** is an annual reporter from 2019-2022, then became inactive and did not report for 2023. Reporter A's ridership data will be displayed for 2019-2022 only.
* **Reporter B** is an annual from 2000-2017, then became inactive and did not report for 2018. Reporter B will be named in the report, but will not display ridership data.
* **Reporter C** was an inactive reporter form 2015-2020, then became an active full reporter for 2021. Reporter C's ridership data will be displayed for 2021-present.


# need to set PUBLIC_FILENAME in update_vars
URL = "https://console.cloud.google.com/storage/" "browser/calitp-publish-data-analysis"

display(
    HTML(
        f"""
        <a href={URL}>
        Download the latest month of data: {PUBLIC_FILENAME}</a>
        """
    )
)

* [annual ridership query](https://github.com/tiffanychu90/curator/blob/use-new-ntd-tables/ntd/ntd_utils.py#L293)
   * includes `agency_status`, what is this? 

In [8]:
full_crosswalk = pd.read_parquet(
    f"{GCS_FILE_PATH}crosswalk.parquet", 
    filesystem=gcsfs.GCSFileSystem(),
).rename(columns = {"ntd_id_2022": "ntd_id"}).drop_duplicates()

In [9]:
# read in data
# annual uses rtpa_name or rtpa_name_split? 
# monthly uses ?
crosswalk = pd.read_parquet(
    f"{GCS_FILE_PATH}crosswalk.parquet", 
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("rtpa_name", "==", rtpa)]]
).rename(columns = {"ntd_id_2022": "ntd_id"}).drop_duplicates()

full_df = pd.read_parquet(f"{GCS_FILE_PATH}annual.parquet",filesystem=gcsfs.GCSFileSystem())

df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    filesystem=gcsfs.GCSFileSystem(),
).merge(
    crosswalk,
    on = "ntd_id",
    how = "inner"
)

In [10]:
df.columns

Index(['key', 'ntd_id', 'mode', 'year', 'type_of_service',
       'unlinked_passenger_trips', 'vehicle_revenue_hours',
       'vehicle_revenue_miles', 'vehicles_operated_in_maxiumum_service',
       'passenger_miles_traveled', 'direction_route_miles',
       'operating_expenses_vehicle_operations',
       'operating_expenses_vehicle_maintenance',
       'operating_expenses_nonvehicle_maintenance',
       'operating_expenses_general_administration', 'operating_expenses_total',
       'fare_revenue', 'opex_per_vrh', 'opex_per_vrm', 'opex_per_upt',
       'upt_per_vrh', 'upt_per_vrm', 'farebox_recovery_ratio', 'agency_status',
       'census_year', 'last_report_year', 'mode_status', 'reporter_type',
       'reporting_module', 'uace_code', 'uza_area_sq_miles',
       'primary_uza_name', 'uza_population', 'source_agency', 'source_city',
       'source_state', 'upt_prior_year', 'upt_change_1yr',
       'upt_pct_change_1yr', 'organization_name', 'county_name', 'rtpa_name',
       'rtpa_name_s

In [11]:
import B3_ntd_utils as ntd_utils

In [12]:
df.source_agency.nunique()
# existing report has 23, where are these 3?

20

In [25]:
full_df[(full_df.source_agency.str.contains("Water Emergency")) |
    (full_df.source_agency.str.contains("County of Sonoma")) |
    (full_df.source_agency.str.contains("MTC"))].groupby("source_agency").unlinked_passenger_trips.sum()

source_agency
County of Sonoma (SCT) - Department of Public Infrastructure - Transit Division          5025472
Metropolitan Transportation Commission (MTC) - Field Operations and Asset Management     4219959
San Francisco Bay Area Water Emergency Transportation Authority (WETA)                  14282199
Name: unlinked_passenger_trips, dtype: Int64

In [26]:
full_df[(full_df.source_agency.str.contains("Water Emergency")) |
    (full_df.source_agency.str.contains("County of Sonoma")) |
    (full_df.source_agency.str.contains("MTC"))].ntd_id.value_counts()

ntd_id
90089    21
90225    14
90094     7
Name: count, dtype: int64

In [27]:
# this is total upt since 2018, which is a parameter in the query
# might need to set this in update_vars, otherwise if it updates, we don't know and caption is wrong
# These counts are totally over counting, more than double, look into why
def proportion_of_upt_by_agency(df: pd.DataFrame):
    initial_agg = (
        df
        .groupby("source_agency")
        .agg(
            total_upt=("unlinked_passenger_trips", "sum")
        ).reset_index()
        .astype({"total_upt": "int64"})
        .sort_values(by="total_upt", ascending=False)
    )
     # % total columns
    initial_agg["pct_of_total_upt"] = ((
        initial_agg["total_upt"] / initial_agg["total_upt"].sum()
    ) * 100).round(decimals=2)

    return initial_agg

In [28]:
# looks like SFMTA is the one that's incorrect - deduping crosswalk solves it

#df.pipe(proportion_of_upt_by_agency)

# missing
# San Francisco Bay Area Water Emergency Transpo.
# County of Sonoma (SCT) - Department of Public 	
# Metropolitan Transportation Commission (MTC) -... 	

In [29]:
# agg by agency, for pie chart
agency_agg_yr = df.pipe(proportion_of_upt_by_agency)
total_upt = agency_agg_yr.total_upt.sum()
agency_count = agency_agg_yr.source_agency.nunique()
# 2,331,534,369

## Report Totals

In [30]:
Markdown(
    f"""
Within {rtpa}:
- Number of Reporters: <b>{agency_count}</b>.
- Total Unlinked Passenger Trips since the beginning of this report: <b>{total_upt:,}</b>.
- Individual Reporters ridership breakdown:
"""
)


Within Metropolitan Transportation Commission:
- Number of Reporters: <b>20</b>.
- Total Unlinked Passenger Trips since the beginning of this report: <b>2,308,006,739</b>.
- Individual Reporters ridership breakdown:


**upt bar chart notes**
* this one has parameter for year
* why does this not use the make_bar_chart function?


In [ ]:
def make_bar_chart_new(
    df: pd.DataFrame, x_col: str, y_col: str, color_col: str, tooltip_cols: list
):
    chart = (
        alt.Chart(df)
        .mark_bar()
        
    )
    return chart

In [20]:
# simple bar chart for total agencies and UPT

def total_upt_chart(df: pd.DataFrame, x_col: str, y_col: str, tool_tip: list):
    bar_chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X(x_col).sort("-y"),
            y=alt.Y(y_col),
            tooltip=tool_tip,
            color=alt.Color(
                x_col,
                title="",
                scale=alt.Scale(
                    range=cp.CALITP_CATEGORY_BRIGHT_COLORS
                    + cp.CALITP_CATEGORY_BOLD_COLORS
                ),
            ),
        )
        .properties(
            title=f"Total Annual Unlinked Passenger Trips per Reporter in RTPA since 2018",
            width=WIDTH,
            height=HEIGHT,
        )
        .resolve_scale(y="independent")
        .interactive()
    )

    return bar_chart

In [ ]:
def make_bar_chart(
    df: pd.DataFrame,
    y_col: str,
    color_col: str,
    title: str,
) -> alt.Chart:

    def short_label(word):
        shorten_dict = {
            "change_1yr": "Change",
            "pct_change_1yr": "Change",
        }
        return shorten_dict[word]

    # For change column, we are missing everything prior to 2023
    # df = df.dropna(subset = y_col)

    # need flag for y_col >,<, 0, missing?
    # count function to how many agencies fall in those categories, then look at those agencies
    # present table

    # x_label = [i for i in df.report_year.unique() if
    #           any(substring in i for substring in
    #               ["-01", "-03", "-06", "-09"])
    #          ]

    chart = (
        (
            alt.Chart(df)
            .mark_bar()
            .encode(
                x=alt.X(
                    "year:O",
                    # axis=alt.Axis(values = x_label),
                    title="Date",
                ),
                y=alt.Y(y_col, title=y_col),
                color=alt.Color(
                    color_col,
                    title="",
                    scale=alt.Scale(
                        range=cp.CALITP_CATEGORY_BRIGHT_COLORS
                        + cp.CALITP_CATEGORY_BOLD_COLORS
                    ),
                ),
                tooltip=["year", y_col, color_col, "rtpa_name"],
            )
            .properties(width=WIDTH, height=HEIGHT)
            .facet(color_col, columns=2, title="")
            .resolve_scale(x="shared", y="independent")
        )
        .properties(title=title)
        .interactive()
    )

    return chart

In [21]:
tooltip_list = ["source_agency", "total_upt", "pct_of_total_upt"]

total_upt_chart(
    agency_agg_yr, 
    x_col="source_agency", 
    y_col="total_upt", 
    tool_tip=tooltip_list
)

NameError: name 'cp' is not defined